# Investigating other USGS-NWIS covariates

See [the waterdata docs](https://doi-usgs.github.io/dataretrieval-python/examples/WaterData_demo.html) for info on data retrieval.

In [1]:
import os
from dataretrieval import waterdata
import sys
sys.path.insert(0, "../src/build/")
import config

API_KEY = "6LDLPN4q8kLDtCqdknbr7OTh0zt0eo1FyaONfoJm"
os.environ['API_USGS_PAT'] = API_KEY

In [29]:
# Table 1: Core Continuous Sensor Parameters for Baseline Modeling
CORE_CONTINUOUS_PCODES = {
    "00010": "temp_water",              # Water temperature (°C)
    "00300": "diss_oxy_con",            # Dissolved oxygen concentration (mg/L)
    "00301": "diss_oxy_sat",            # Dissolved oxygen saturation (%)
    "00400": "ph",                      # pH (standard units)
    "00095": "spec_cond",               # Specific conductance (µS/cm at 25°C)
    "00060": "discharge",               # Stream discharge (ft³/s)
    "00065": "stage",                   # Gage height / stage (ft)
}

# Table 2: Expanded Continuous Sensor Parameters for Advanced Modeling
EXPANDED_CONTINUOUS_PCODES = {
    "61028": "turbidity_ntu",           # Turbidity (NTU)
    "63680": "turbidity_fnu",           # Turbidity (FNU)
    "32295": "fdom_qse",                # Colored Dissolved Organic Matter (QSE)
    "32322": "fdom_ppb",                # Colored Dissolved Organic Matter (ppb QSE)
    "31883": "chlorophyll_a",           # Chlorophyll a, fluorescence (µg/L)
    "32320": "chlorophyll_rfu",         # Chlorophyll fluorescence (RFU)
    "32321": "phycocyanin_rfu",         # Phycocyanin fluorescence (RFU)
    "00045": "precipitation",           # Total precipitation (inches)
}

# Table 3: Discrete Sample Parameters for Model Training & Spatial Vulnerability
DISCRETE_SAMPLE_PCODES = {
    "00631": "nitrate_nitrite_diss",    # Nitrate plus nitrite, dissolved (mg/L as N)
    "00625": "tkn_unfiltered",          # Total Kjeldahl Nitrogen (mg/L as N)
    "62854": "total_nitrogen_diss",     # Total Nitrogen, dissolved (mg/L)
    "80154": "suspended_sediment_conc", # Suspended-Sediment Concentration (mg/L)
    "00681": "organic_carbon_diss",     # Dissolved Organic Carbon (DOC) (mg/L)
    "00940": "chloride_diss",           # Chloride, dissolved (mg/L)
    "00955": "silica_diss",             # Silica, dissolved (mg/L as SiO2)
}

ALL_CODES = list(CORE_CONTINUOUS_PCODES.keys()) + list(EXPANDED_CONTINUOUS_PCODES.keys()) + list(DISCRETE_SAMPLE_PCODES.keys())

In [61]:
# get metadata on sites within bounding box
bbox = config.get_region_bbox()
print(bbox)
data = waterdata.get_time_series_metadata(parameter_code="63680", bbox=bbox)
sitelist = data[0].monitoring_location_id.unique().tolist()
sites = waterdata.get_monitoring_locations(monitoring_location_id=sitelist)
site_type_codes = waterdata.get_codes(code_service="sitetype")

(-97.6, 39.8, -89.5, 44.5)


Retrieving: time-series-metadata · 1 page · 216 rows · 997/1,000 requests remaining
Retrieving: monitoring-locations · 1 page · 68 rows · 997/1,000 requests remaining


In [28]:
# display unique site types
print(sites[0].site_type.unique().tolist())
for d in sites[0][["monitoring_location_id", "site_type"]].groupby(["site_type"]):
    print(d)

# print legible info for one row of the site metadata dict
d = sites[0].head(1).to_dict()
d = {k : v[0] for k,v in d.items()} # reformat values so they are not dicts
for key, value in d.items():
    print(key, " : ", value) 

['Stream', 'Storm sewer', 'Well']
(('Storm sewer',),    monitoring_location_id    site_type
13          USGS-05406503  Storm sewer)
(('Stream',),    monitoring_location_id site_type
0           USGS-05320270    Stream
1           USGS-05321995    Stream
2           USGS-05326180    Stream
3           USGS-05326189    Stream
4           USGS-05374900    Stream
..                    ...       ...
63   USGS-410546096194301    Stream
64   USGS-411105095532301    Stream
65   USGS-411632096020701    Stream
66   USGS-411829096004801    Stream
67   USGS-412126095565201    Stream

[64 rows x 2 columns])
(('Well',),    monitoring_location_id site_type
54   USGS-401913089534501      Well
55   USGS-401934089541703      Well
56   USGS-401934089541704      Well)
monitoring_location_id  :  USGS-05320270
geometry  :  POINT (-93.9085628225437 43.9966325374185)
agency_code  :  USGS
agency_name  :  U.S. Geological Survey
monitoring_location_number  :  05320270
monitoring_location_name  :  LITTLE COBB RIV

In [31]:
# get metadata on sites within bounding box
bbox = config.get_region_bbox()
ac_data = waterdata.get_time_series_metadata(parameter_code=ALL_CODES, bbox=bbox)
ac_sitelist = ac_data[0].monitoring_location_id.unique().tolist()
ac_sites = waterdata.get_monitoring_locations(monitoring_location_id=ac_sitelist)
site_type_codes = waterdata.get_codes(code_service="sitetype")

Retrieving: time-series-metadata · 1 page · 6,629 rows · 990/1,000 requests remaining
Retrieving: monitoring-locations · chunk 4/4 · 4 pages · 1,552 rows · 990/1,000 requests remaining


In [ ]:
# 1552
print("Unique sites: ", len(ac_sites[0].monitoring_location_id.unique()))

# Atmosphere; Ditch; Field, Pasture, Orchard or Nursery;
# Groundwater drain; Lake, Reservoir, Impoundment;
# Land; Multiple wells; Outfall; Spring; Storm sewer;
# Stream; Test hole not completed as a well; Well
# 1275 are Stream sites
print(ac_sites[0][["site_type", "monitoring_location_id"]].groupby(["site_type"]).describe())
atmosphere_sites = ac_sites[0].loc[ac_sites[0].site_type == "Atmosphere"]

print(atmosphere_sites.head())

Unique sites:  1552
                                    monitoring_location_id         \
                                                     count unique   
site_type                                                           
Atmosphere                                              89     89   
Ditch                                                   14     14   
Field, Pasture, Orchard, or Nursery                     16     16   
Groundwater drain                                        2      2   
Lake, Reservoir, Impoundment                            43     43   
Land                                                    13     13   
Multiple wells                                           1      1   
Outfall                                                  2      2   
Spring                                                   2      2   
Storm sewer                                             11     11   
Stream                                                1275   1275   
Test hole not 

In [60]:
from dataretrieval import wqp

# Fetch discrete nitrate results across Iowa using MM-DD-YYYY date strings
df, metadata = wqp.get_results(
    statecode="US:19",
    characteristicName="Nitrate",
    startDateLo="01-01-2023",  # MM-DD-YYYY format required by WQP
    startDateHi="12-31-2023"   # MM-DD-YYYY format required by WQP
)

print(f"Successfully retrieved {len(df)} records.")

# these are the two activity types
df_field = df[df.ActivityTypeCode == "Field Msr/Obs"]
df_routine = df[df.ActivityTypeCode == "Sample-Routine"]

print(df["SampleCollectionMethod/MethodName"].unique())

d = df_routine.head(1).to_dict()
d = {k : v[82] for k, v in d.items()}
for k,v in d.items():
    print(k, " : ", v)
    

Successfully retrieved 1779 records.
<ArrowStringArray>
[                 nan, 'Grab sample  (dip)',    'Weighted bottle',
 'Multiple verticals',   'Submersible pump',   'Peristaltic pump',
   'NRSA Grab Sample']
Length: 7, dtype: str
OrganizationIdentifier  :  USGS-IA
OrganizationFormalName  :  USGS Iowa Water Science Center
ActivityIdentifier  :  nwisia.01.02300299
ActivityTypeCode  :  Sample-Routine
ActivityMediaName  :  Water
ActivityMediaSubdivisionName  :  Surface Water
ActivityStartDate  :  2023-01-17
ActivityStartTime/Time  :  09:20:00
ActivityStartTime/TimeZoneCode  :  CST
ActivityEndDate  :  nan
ActivityEndTime/Time  :  nan
ActivityEndTime/TimeZoneCode  :  nan
ActivityDepthHeightMeasure/MeasureValue  :  nan
ActivityDepthHeightMeasure/MeasureUnitCode  :  nan
ActivityDepthAltitudeReferencePointText  :  nan
ActivityTopDepthHeightMeasure/MeasureValue  :  nan
ActivityTopDepthHeightMeasure/MeasureUnitCode  :  nan
ActivityBottomDepthHeightMeasure/MeasureValue  :  nan
ActivityBottomD

In [ ]:
# Use the large_bbox to get the whole cornbelt
# right now it is omitted to see every nitrate sensor in the usgs database
large_bbox = (-105.908203,36.456636,-83.803711,48.370848)
data = waterdata.get_time_series_metadata(parameter_code="99133")
sitelist = data[0].monitoring_location_id.unique().tolist()
sites = waterdata.get_monitoring_locations(monitoring_location_id=sitelist)
site_type_codes = waterdata.get_codes(code_service="sitetype")

Retrieving: time-series-metadata · 1 page · 860 rows · 988/1,000 requests remaining
Retrieving: monitoring-locations · 1 page · 311 rows · 987/1,000 requests remaining


In [67]:
data[0].monitoring_location_id.unique()

<ArrowStringArray>
[       'USGS-02035000',        'USGS-02248380', 'USGS-293746098265401',
        'USGS-11455478',        'USGS-01645762',        'USGS-05418400',
        'USGS-05567500', 'USGS-292618099165901',        'USGS-11455385',
        'USGS-01574000',
 ...
 'USGS-382006121401601',        'USGS-01577500',        'USGS-07291000',
        'USGS-01474500',        'USGS-06902000', 'USGS-413846070480601',
        'USGS-03123499',        'USGS-07289730', 'USGS-383114121350601',
        'USGS-07144780']
Length: 311, dtype: str

In [4]:
import sys
sys.path.insert(0, "..")
from src.data.access import get_data, get_comid_attributes, get_agtile

site = "WQS0039"
data = get_data(site)
print(data.comid_attrs)
print(data.agtile)

{'cat_tiles92': 19.5623, 'tot_tiles92': 33.2308, 'cat_bfi': 43.0, 'tot_bfi': 44.72, 'cat_contact': 683.99, 'tot_contact': 677.05}
    node_id  global_node_id  tile_cells  ag_cells
0         0           13728         NaN       NaN
1         1           13729         NaN       NaN
2         2           13899         NaN       NaN
3         3           13900         NaN       NaN
4         4           13901         NaN       NaN
..      ...             ...         ...       ...
73       73           15459         NaN       NaN
74       74           15629         NaN       NaN
75       75           15630         NaN       NaN
76       76           15631         NaN       NaN
77       77           15632         NaN       NaN

[78 rows x 4 columns]
